In [1]:
# Load the autoreload extension
%load_ext autoreload

# Set autoreload mode
%autoreload 2

# Label Mapping

In [2]:
import requests
import pandas as pd

The Label Mapping Endpoint recently had another 125 new maps added  

In [3]:
mapping_endpoint ="https://api.datamermaid.org/v1/classification/labelmappings/?provider=CoralNet" 

response = requests.get(mapping_endpoint)
data = response.json()
labelset = data["results"]

while data["next"]:
    response = requests.get(data["next"])
    data = response.json()
    labelset.extend(data["results"])
label_mapping = {
    label["provider_id"]: label["benthic_attribute_name"] for label in labelset
}

In [4]:
df_mapping_orig = pd.read_csv("dataframes/CoralNetMermaidMatchedCoralFocusModel2Reassign_20240503.csv")
df_mapping_upd = pd.read_csv("dataframes/class_mapping_concepts_2026_04_28.csv")

## Original Mapping
https://docs.google.com/spreadsheets/d/1BUk4hK78CxwbFWLSeu6taFDJws1z6e96WTNW6E8vmWo/edit?gid=1827266844#gid=1827266844

The original label mapping dictionary consists of 711 (710 in the endpoint) CoralNet labels (IDs) that have been mapped to 305 unique benthic attributes, 12 unique growth forms for a total of 344 unique benthic attribute - growth form combinations.

When using the focus labels (CoralFocus3Label), there are 55 unique BA+GF combinations (+1 remove label), with 119 labels being removed from the final mapping.

In [5]:
df_mapping_orig.head(3)

,CoralNetID,CoralNetName,Default short code,Functional group,Public sources using,Public annotation count,NewCoralNetID,Top100,MERMAID_BA,BA_AssignNotes,...,CoralFocusLabel,precision_CoralFocusModel,recall_CoralFocusModel,f1-score_CoralFocusModel,support_CoralFocusModel,CoralFocus2Label,CoralFocus3Label,CoralFocus3_BaID,CoralFocus3_GfID,Unnamed: 44
0,82,Turf algae,Turf,Algae,321,5496896,82,1,Turf algae,NaN,...,Turf algae,0.605677,0.356949,0.449179,9923,Turf algae,Turf algae,20090bf4-868e-431b-974c-ab9be5bbdb5f,NaN,NaN
1,84,Sand,Sand,Soft Substrate,418,1882701,84,1,Sand,NaN,...,Sand,0.749551,0.706228,0.727245,10052,Sand,Sand,b76bca12-884b-4404-bb9f-97d505b0fe58,NaN,NaN
2,1348,CRED-Turf growing on hard substrate,*TURFH,Hard Substrate,18,1216402,82,1,Turf algae,NaN,...,Turf algae,0.605677,0.356949,0.449179,9923,Turf algae,Turf algae,20090bf4-868e-431b-974c-ab9be5bbdb5f,NaN,NaN


In [6]:
len(label_mapping), df_mapping_orig.shape

(836, (711, 45))

In [7]:
df_mapping_orig["MERMAID_BA"].nunique(), df_mapping_orig["MERMAID_GF"].nunique(), df_mapping_orig["ba_gf"].nunique()

(305, 12, 344)

In [8]:
print("Number of labels who have been removed from the final mapping:", df_mapping_orig[df_mapping_orig["CoralFocus3Label"]=="Remove"].shape[0])

Number of labels who have been removed from the final mapping: 119


In [9]:
df_mapping_orig["CoralFocusLabel"].nunique(), df_mapping_orig["CoralFocus2Label"].nunique(), df_mapping_orig["CoralFocus3Label"].nunique()

(70, 63, 56)

## New Label Mapping (Jonathan's)
https://docs.google.com/spreadsheets/d/1DjE7SpXE2AaKsJXbqNm18ClGIGp5nAmwO9LhmG9yzTQ/edit?usp=sharing

The updated label mapping dictionary consists of 1320 unique labels from three sources (778 from MERMAID, 711 from CoralNet, and 95 from Coralscapes, resulting in a total of 1584 labels) that have been mapped to 896 unique labels. 

In [10]:
df_mapping_upd.head(3)

,name,source,should_map_to_label,should_map_to_label_source,kingdom,phylum,order,family,genus,species,...,dead,bleached,algae,background,anthropogenic,trash,transect,macroalgae,dark,class
0,acropora alive,coralscapes,NaN,NaN,animalia,cnidaria,scleractinia,acroporidae,acropora,not_given,...,False,False,False,False,False,False,False,False,False,hexacorallia
1,acropora bleached,coralscapes,NaN,NaN,animalia,cnidaria,scleractinia,acroporidae,acropora,not_given,...,False,True,False,False,False,False,False,False,False,hexacorallia
2,acropora dead,coralscapes,NaN,NaN,animalia,cnidaria,scleractinia,acroporidae,acropora,not_given,...,True,False,False,False,False,False,False,False,False,hexacorallia


In [11]:
df_mapping_upd.shape, df_mapping_upd["name"].nunique()

((1872, 44), 1516)

In [12]:
df_mapping_upd["source"].value_counts()

source
mermaid                   778
coralnet                  713
catlin_seaview            194
coralscapes                95
mosaics_ucsd               34
moorea_labeled_corals      30
pacific_labeled_corals     20
benthos_yuval               8
Name: count, dtype: int64

In [13]:
tmp = df_mapping_upd["name"].copy()
tmp = tmp.mask(df_mapping_upd["should_map_to_label"].notna(), df_mapping_upd["should_map_to_label"])
print("The updated number of unique labels in the mapping is", tmp.nunique())

The updated number of unique labels in the mapping is 908


## Differences between original and updated coral mapping

In [14]:
tmp = df_mapping_upd[df_mapping_upd["source"]=="coralnet"][["name", "should_map_to_label", "should_map_to_label_source"]].copy()
tmp2 = df_mapping_orig[["CoralNetName", "ba_gf", "CoralFocus3Label"]]
tmp2 = tmp2.apply(lambda s: s.str.lower() if s.dtype == "object" else s)
df_mapping_comparison = tmp.merge(tmp2, left_on = "name", right_on = "CoralNetName", how = "outer")

In [15]:
df_mapping_comparison.head(3)

,name,should_map_to_label,should_map_to_label_source,CoralNetName,ba_gf,CoralFocus3Label
0,acanthastrea,acanthastrea,mermaid,acanthastrea,acanthastrea,remove
1,acanthophora spicifera,acanthophora spicifera,mermaid,acanthophora spicifera,acanthophora spicifera,macroalgae
2,acetabularia spp.,acetabularia,mermaid,acetabularia spp.,acetabularia,macroalgae


In [16]:
print("Number of labels that have the same mapping in the original and updated mapping:",)
print((df_mapping_comparison["should_map_to_label"]== df_mapping_comparison["ba_gf"]).value_counts())

Number of labels that have the same mapping in the original and updated mapping:
True     406
False    315
Name: count, dtype: int64


In [17]:
df_mapping_comparison[(df_mapping_comparison["should_map_to_label"]!=df_mapping_comparison["ba_gf"])]

,name,should_map_to_label,should_map_to_label_source,CoralNetName,ba_gf,CoralFocus3Label
3,acropora,acropora alive,coralscapes,acropora,acropora,acropora
4,acropora (arborescent table),table acropora alive,coralscapes,acropora (arborescent table),acropora arborescent,remove
5,acropora (arborescent),acropora alive,coralscapes,acropora (arborescent),acropora arborescent,remove
6,acropora (branching),acropora alive,coralscapes,acropora (branching),acropora branching,acropora - branching
7,acropora (corymbose),acropora alive,coralscapes,acropora (corymbose),acropora corymbose,remove
...,...,...,...,...,...,...
710,wand,transect tools,coralscapes,wand,other,remove
711,water,background,coralscapes,water,other,remove
712,worms: polychaetes,polychaeta,coralnet,worms: polychaetes,other invertebrates,other invertebrates
713,worms: polychaetes: tube worms,polychaeta,coralnet,worms: polychaetes: tube worms,other invertebrates,other invertebrates


In [18]:
print("Number of labels that have the same mapping in the original and updated mapping (using the CoralFocus3Label column):",)
print((df_mapping_comparison["should_map_to_label"]== df_mapping_comparison["CoralFocus3Label"]).value_counts())

Number of labels that have the same mapping in the original and updated mapping (using the CoralFocus3Label column):
False    614
True     107
Name: count, dtype: int64


In [19]:
print("There are 23 labels that were previously mapped to other that now have an associated class")
df_mapping_comparison[(df_mapping_comparison["ba_gf"]=="other") & 
                      (df_mapping_comparison["should_map_to_label"].notna())& 
                      (df_mapping_comparison["should_map_to_label"]!="unknown")].shape

There are 23 labels that were previously mapped to other that now have an associated class


(23, 6)

# Concept EDA

In [20]:
df_mapping_orig.head(3)

,CoralNetID,CoralNetName,Default short code,Functional group,Public sources using,Public annotation count,NewCoralNetID,Top100,MERMAID_BA,BA_AssignNotes,...,CoralFocusLabel,precision_CoralFocusModel,recall_CoralFocusModel,f1-score_CoralFocusModel,support_CoralFocusModel,CoralFocus2Label,CoralFocus3Label,CoralFocus3_BaID,CoralFocus3_GfID,Unnamed: 44
0,82,Turf algae,Turf,Algae,321,5496896,82,1,Turf algae,NaN,...,Turf algae,0.605677,0.356949,0.449179,9923,Turf algae,Turf algae,20090bf4-868e-431b-974c-ab9be5bbdb5f,NaN,NaN
1,84,Sand,Sand,Soft Substrate,418,1882701,84,1,Sand,NaN,...,Sand,0.749551,0.706228,0.727245,10052,Sand,Sand,b76bca12-884b-4404-bb9f-97d505b0fe58,NaN,NaN
2,1348,CRED-Turf growing on hard substrate,*TURFH,Hard Substrate,18,1216402,82,1,Turf algae,NaN,...,Turf algae,0.605677,0.356949,0.449179,9923,Turf algae,Turf algae,20090bf4-868e-431b-974c-ab9be5bbdb5f,NaN,NaN


In [21]:
df_mapping_upd.head(3)

,name,source,should_map_to_label,should_map_to_label_source,kingdom,phylum,order,family,genus,species,...,dead,bleached,algae,background,anthropogenic,trash,transect,macroalgae,dark,class
0,acropora alive,coralscapes,NaN,NaN,animalia,cnidaria,scleractinia,acroporidae,acropora,not_given,...,False,False,False,False,False,False,False,False,False,hexacorallia
1,acropora bleached,coralscapes,NaN,NaN,animalia,cnidaria,scleractinia,acroporidae,acropora,not_given,...,False,True,False,False,False,False,False,False,False,hexacorallia
2,acropora dead,coralscapes,NaN,NaN,animalia,cnidaria,scleractinia,acroporidae,acropora,not_given,...,True,False,False,False,False,False,False,False,False,hexacorallia


In [22]:
coralnet_labels_source_df = pd.read_csv("dataframes/coralnet_per_source_labels_annotations.csv").drop("Unnamed: 0", axis=1)
coralnet_labels_source_df = (
    coralnet_labels_source_df
    .groupby("Label ID", as_index=True)
    .agg(
        **{
            "Num Annotations": ("Num Annotations", "sum"),
            "Num Images": ("Num Images", "sum"),
            "Num Sources": ("Source ID", "nunique"),
        }
    )
)

In [23]:
coralnet_id2_name = df_mapping_orig[["CoralNetID", "CoralNetName"]].drop_duplicates().set_index("CoralNetID")["CoralNetName"].to_dict()
coralnet_labels_source_df = coralnet_labels_source_df.reset_index()
coralnet_labels_source_df["CoralNetName"] = coralnet_labels_source_df["Label ID"].map(coralnet_id2_name)
coralnet_labels_source_df = coralnet_labels_source_df[coralnet_labels_source_df["CoralNetName"].notna()]
coralnet_labels_source_df["name"] = coralnet_labels_source_df["CoralNetName"].str.lower()

In [24]:
coralnet_labels_source_df = coralnet_labels_source_df.merge(df_mapping_upd[df_mapping_upd["source"]=="coralnet"], on = "name", how = "left")

In [25]:
coralnet_labels_source_df.head(5)

,Label ID,Num Annotations,Num Images,Num Sources,CoralNetName,name,source,should_map_to_label,should_map_to_label_source,kingdom,...,dead,bleached,algae,background,anthropogenic,trash,transect,macroalgae,dark,class
0,58,2521,1763,77,Acanthastrea,acanthastrea,coralnet,acanthastrea,mermaid,animalia,...,not given,not given,False,False,False,False,False,False,False,hexacorallia
1,59,418545,72003,148,Acropora,acropora,coralnet,acropora alive,coralscapes,animalia,...,not given,not given,False,False,False,False,False,False,False,hexacorallia
2,60,18932,9101,111,Astreopora,astreopora,coralnet,astreopora,mermaid,animalia,...,not given,not given,False,False,False,False,False,False,False,hexacorallia
3,61,9115,5054,99,Cyphastrea,cyphastrea,coralnet,cyphastrea,mermaid,animalia,...,not given,not given,False,False,False,False,False,False,False,hexacorallia
4,62,8614,4900,38,Favia (Indo-Pacific),favia (indo-pacific),coralnet,favia,mermaid,animalia,...,not given,not given,False,False,False,False,False,False,False,hexacorallia


In [26]:
coralnet_labels_source_df.columns

Index(['Label ID', 'Num Annotations', 'Num Images', 'Num Sources',
       'CoralNetName', 'name', 'source', 'should_map_to_label',
       'should_map_to_label_source', 'kingdom', 'phylum', 'order', 'family',
       'genus', 'species', 'oval', 'arborescent', 'encrusting', 'digitate',
       'meandroid', 'columnar', 'free_living', 'plating', 'fleshy',
       'submassive', 'round', 'massive', 'tubular', 'bushy', 'external_polyps',
       'foliose', 'solitary', 'brain', 'phaceloid', 'branching', 'tabular',
       'corymbose', 'lobed_brain', 'cup_coral', 'dead', 'bleached', 'algae',
       'background', 'anthropogenic', 'trash', 'transect', 'macroalgae',
       'dark', 'class'],
      dtype='object')

In [27]:
for taxonomic_rank in ["kingdom", "phylum", "class", "order", "family", "genus"]:
    taxonomic_counts = (
        coralnet_labels_source_df.assign(**{taxonomic_rank: coralnet_labels_source_df[taxonomic_rank].fillna("unknown")})
        .groupby(taxonomic_rank, as_index=False)
        .agg(
            num_annotations=("Num Annotations", "sum"),
            num_images=("Num Images", "sum"),
        )
        .sort_values("num_annotations", ascending=False)
    )
    display(taxonomic_counts.head(5))

,kingdom,num_annotations,num_images
4,none,18322158,1874890
0,animalia,7143524,1554262
5,not_given,2827395,675847
6,plantae,1210150,163909
1,bacillati,324223,84221


,phylum,num_annotations,num_images
10,none,18322158,1874890
6,cnidaria,5485773,1367720
16,rhodophyta,2248200,537239
9,mollusca,852580,30913
11,not_given,680663,47308


,class,num_annotations,num_images
15,none,18322158,1874890
11,hexacorallia,4494156,1154852
9,florideophyceae,2248200,537239
16,not_given,1746778,282444
2,bivalvia,845343,28018


,order,num_annotations,num_images
32,none,18322158,1874890
40,scleractinia,4026044,1090713
33,not_given,2733301,354065
14,corallinales,1924402,416476
2,alismatales,667243,40226


,family,num_annotations,num_images
57,none,18322158,1874890
58,not_given,5782141,926313
69,poritidae,1210890,288370
1,acroporidae,1141836,275971
29,dictyotaceae,545069,132807


,genus,num_annotations,num_images
109,none,18322158,1874890
110,not_given,5962311,967323
134,porites,1199552,284589
4,acropora,541128,102281
100,montipora,506017,144198


In [28]:
coralnet_labels_source_df.columns

Index(['Label ID', 'Num Annotations', 'Num Images', 'Num Sources',
       'CoralNetName', 'name', 'source', 'should_map_to_label',
       'should_map_to_label_source', 'kingdom', 'phylum', 'order', 'family',
       'genus', 'species', 'oval', 'arborescent', 'encrusting', 'digitate',
       'meandroid', 'columnar', 'free_living', 'plating', 'fleshy',
       'submassive', 'round', 'massive', 'tubular', 'bushy', 'external_polyps',
       'foliose', 'solitary', 'brain', 'phaceloid', 'branching', 'tabular',
       'corymbose', 'lobed_brain', 'cup_coral', 'dead', 'bleached', 'algae',
       'background', 'anthropogenic', 'trash', 'transect', 'macroalgae',
       'dark', 'class'],
      dtype='object')

In [29]:
# for growth_form in ['massive', 'brain', 'submassive', 'corymbose',
#        'round', 'tubular', 'free_living', 'tabular', 'external_polyps',
#        'plating', 'lobed_brain', 'cup_coral', 'fleshy', 'columnar', 'solitary',
#        'prostrate', 'branching', 'meandroid', 'encrusting', 'arborescent',
#        'bushy', 'digitate', 'phaceloid', 'colonial', 'foliose']:

for growth_form in ['arborescent', 'encrusting', 'digitate',
       'meandroid', 'columnar', 'free_living', 'plating', 'fleshy',
       'submassive', 'round', 'massive', 'tubular', 'bushy', 'external_polyps',
       'foliose', 'solitary', 'brain', 'phaceloid', 'branching', 'tabular',
       'corymbose', 'lobed_brain', 'cup_coral']:
    
    growth_form_counts = (
        coralnet_labels_source_df.assign(**{growth_form: coralnet_labels_source_df[growth_form].fillna("unknown")})
        .groupby(growth_form, as_index=False)
        .agg(
            num_annotations=("Num Annotations", "sum"),
            num_images=("Num Images", "sum"),
        )
        # .sort_values("num_annotations", ascending=False)
        )
    print(growth_form)
    display(growth_form_counts.head(5))

arborescent


,arborescent,num_annotations,num_images
0,False,5283,2151
1,True,139873,16064
2,not given,29692297,4337707
3,unknown,52141,13581


encrusting


,encrusting,num_annotations,num_images
0,False,2649,1167
1,True,397658,119492
2,not given,29437146,4235263
3,unknown,52141,13581


digitate


...

,corymbose,num_annotations,num_images
0,False,19295,5911
1,not given,29818158,4350011
2,unknown,52141,13581


lobed_brain


,lobed_brain,num_annotations,num_images
0,False,376,224
1,True,8169,4301
2,not given,29828908,4351397
3,unknown,52141,13581


cup_coral


,cup_coral,num_annotations,num_images
0,False,194307,34275
1,True,10,9
2,not given,29643136,4321638
3,unknown,52141,13581


In [30]:
for concept in ['dead', 'bleached', 'algae',
       'background', 'anthropogenic', 'trash', 'transect', 'macroalgae',
       'dark', 'class']:
    concept_counts = (
        coralnet_labels_source_df.assign(**{concept: coralnet_labels_source_df[concept].fillna("unknown")})
        .groupby(concept, as_index=False)
        .agg(
            num_annotations=("Num Annotations", "sum"),
            num_images=("Num Images", "sum"),
        )
        # .sort_values("num_annotations", ascending=False)
        )
    print(concept)
    display(concept_counts.head(5))

dead


,dead,num_annotations,num_images
0,False,127478,27214
1,True,611197,67800
2,not given,29098778,4260908
3,unknown,52141,13581


bleached


,bleached,num_annotations,num_images
0,False,193399,34144
1,True,70024,16848
2,not given,29574030,4304930
3,unknown,52141,13581


algae


...

,macroalgae,num_annotations,num_images
0,False,24824645,3662464
1,True,3141170,360261
2,not given,1871638,333197
3,unknown,52141,13581


dark


,dark,num_annotations,num_images
0,False,29643570,4265173
1,True,193883,90749
2,unknown,52141,13581


class


,class,num_annotations,num_images
0,ascidiacea,10076,2622
1,asteroidea,1637,1311
2,bivalvia,845343,28018
3,chrysophyceae,13,13
4,crinoidea,3,2


In [31]:
for concept in ['dead', 'bleached']:
    concept_counts = (
        coralnet_labels_source_df.assign(**{concept: coralnet_labels_source_df[concept].fillna("unknown")})
        .groupby(concept, as_index=False)
        .agg(
            num_annotations=("Num Annotations", "sum"),
            num_images=("Num Images", "sum"),
        )
        # .sort_values("num_annotations", ascending=False)
        )
    print(concept)
    display(concept_counts.head(5))

dead


,dead,num_annotations,num_images
0,False,127478,27214
1,True,611197,67800
2,not given,29098778,4260908
3,unknown,52141,13581


bleached


,bleached,num_annotations,num_images
0,False,193399,34144
1,True,70024,16848
2,not given,29574030,4304930
3,unknown,52141,13581


# Final Label Set

In [32]:
df_labelset100 = pd.read_csv("dataframes/mapped_to_mermaid_attributes.csv")
df_labelset100_top = df_labelset100[df_labelset100["top100"]=="1"]

In [33]:
df_concept_mapping = pd.read_csv("dataframes/class_mapping_concepts_2026_04_28.csv")

In [34]:
taxonomic_concept_columns = ["species", "genus", "family", "order", "class", "phylum", "kingdom"]
morphologic_concept_columns = [ 'oval',
       'arborescent', 'encrusting', 'digitate', 'meandroid', 'columnar',
       'free_living', 'plating', 'fleshy', 'submassive', 'round', 'massive',
       'tubular', 'bushy', 'external_polyps', 'foliose', 'solitary', 'brain',
       'phaceloid', 'branching', 'tabular', 'corymbose', 'lobed_brain',
       'cup_coral']
health_concept_columns = ['dead', 'bleached']
noncoral_concept_columns = ['algae', 'background', 'anthropogenic',
       'trash', 'transect', 'macroalgae', 'dark']

In [35]:
df_concept_mapping.head(3)

,name,source,should_map_to_label,should_map_to_label_source,kingdom,phylum,order,family,genus,species,...,dead,bleached,algae,background,anthropogenic,trash,transect,macroalgae,dark,class
0,acropora alive,coralscapes,NaN,NaN,animalia,cnidaria,scleractinia,acroporidae,acropora,not_given,...,False,False,False,False,False,False,False,False,False,hexacorallia
1,acropora bleached,coralscapes,NaN,NaN,animalia,cnidaria,scleractinia,acroporidae,acropora,not_given,...,False,True,False,False,False,False,False,False,False,hexacorallia
2,acropora dead,coralscapes,NaN,NaN,animalia,cnidaria,scleractinia,acroporidae,acropora,not_given,...,True,False,False,False,False,False,False,False,False,hexacorallia


In [36]:
df_concept_mapping["source"].value_counts()

source
mermaid                   778
coralnet                  713
catlin_seaview            194
coralscapes                95
mosaics_ucsd               34
moorea_labeled_corals      30
pacific_labeled_corals     20
benthos_yuval               8
Name: count, dtype: int64

In [37]:
concept_mapping_dict = {}
for _, row in df_concept_mapping.iterrows():
    source_label = row["name"]
    source = row["source"]
    taxonomic_concepts = row[taxonomic_concept_columns].tolist()
    morphologic_concepts = row[morphologic_concept_columns].tolist()
    health_concepts = row[health_concept_columns].tolist()
    noncoral_concepts = row[noncoral_concept_columns].tolist()
    concept_mapping_dict[source, source_label] = {
        "taxonomic_concepts": taxonomic_concepts,
        "morphologic_concepts": morphologic_concepts,
        "health_concepts": health_concepts,
        "noncoral_concepts": noncoral_concepts
    }

In [38]:
counter = 0
for MERMAID_label in df_labelset100_top["name"].values:
    if ("mermaid", MERMAID_label.lower()) not in concept_mapping_dict:
        print(f"MERMAID label '{MERMAID_label}' does not have a mapping in the concept mapping dataframe.")
        counter += 1
print(f"Total number of MERMAID labels (Priority Top 100) without a mapping: {counter}")

Total number of MERMAID labels (Priority Top 100) without a mapping: 0


In [39]:
df_labelset100.head(1)

,id,created on,created by,updated on,updated by,status,name,CoralNetAnnotations,top100,priority,priority_notes,parent,Life history list,Region list
0,20090bf4-868e-431b-974c-ab9be5bbdb5f,2018-04-04 19:03:43.852250+00:00,NaN,2022-08-10 12:42:00.960645+00:00,NaN,superuser approved,Turf algae,7443962,1,1.0,top level category,NaN,NaN,"Central Indo-Pacific,Eastern Indo-Pacific,Trop..."


In [40]:
counter = 0
annotations_count = 0
for _, row in df_labelset100.iterrows():
    MERMAID_label = row["name"]
    annotations_count = row["CoralNetAnnotations"]
    if ("mermaid", MERMAID_label.lower()) not in concept_mapping_dict:
        print(f"MERMAID label '{MERMAID_label}' with {annotations_count} annotations does not have a mapping in the concept mapping dataframe.")
        counter += 1
        annotations_count += row["CoralNetAnnotations"]
print(f"Total number of MERMAID labels without a mapping: {counter} for a total of {annotations_count} annotations")

MERMAID label 'Aaptos' with 0 annotations does not have a mapping in the concept mapping dataframe.
MERMAID label 'Acanthogorgia' with 0 annotations does not have a mapping in the concept mapping dataframe.
MERMAID label 'Algal assemblage' with 0 annotations does not have a mapping in the concept mapping dataframe.
MERMAID label 'Atriolum' with 0 annotations does not have a mapping in the concept mapping dataframe.
MERMAID label 'Atriolum robustum' with 0 annotations does not have a mapping in the concept mapping dataframe.
MERMAID label 'Botryllus' with 0 annotations does not have a mapping in the concept mapping dataframe.
MERMAID label 'Chironephthya' with 0 annotations does not have a mapping in the concept mapping dataframe.
MERMAID label 'Clathria (Microciona)' with 0 annotations does not have a mapping in the concept mapping dataframe.
MERMAID label 'Clathria (Microciona) mima' with 0 annotations does not have a mapping in the concept mapping dataframe.
MERMAID label 'Dead Coral

In [41]:
from mermaidseg.datasets.concepts import initialize_benthic_hierarchy
hierarchy_dict = initialize_benthic_hierarchy()

In [46]:
counter = 0
for MERMAID_label in set(hierarchy_dict.keys()):
    if ("mermaid", MERMAID_label.lower()) not in concept_mapping_dict:
        print(f"MERMAID label '{MERMAID_label}' does not have a mapping in the concept mapping dataframe.")
        counter += 1
print(f"Total number of MERMAID labels without a mapping: {counter} annotations")

MERMAID label 'Chironephthya' does not have a mapping in the concept mapping dataframe.
MERMAID label 'Pachyrhynchia' does not have a mapping in the concept mapping dataframe.
MERMAID label 'Acanthogorgia' does not have a mapping in the concept mapping dataframe.
MERMAID label 'Styelidae' does not have a mapping in the concept mapping dataframe.
MERMAID label 'Tridacna maxima' does not have a mapping in the concept mapping dataframe.
MERMAID label 'Tridacna gigas' does not have a mapping in the concept mapping dataframe.
MERMAID label 'Euryspongia' does not have a mapping in the concept mapping dataframe.
MERMAID label 'Neopetrosia chaliniformis' does not have a mapping in the concept mapping dataframe.
MERMAID label 'Atriolum robustum' does not have a mapping in the concept mapping dataframe.
MERMAID label 'Botryllus' does not have a mapping in the concept mapping dataframe.
MERMAID label 'Radianthus magnifica' does not have a mapping in the concept mapping dataframe.
MERMAID label 'C

In [5]:
morphologic_concept_columns = [ 'oval',
       'arborescent', 'encrusting', 'digitate', 'meandroid', 'columnar',
       'free_living', 'plating', 'fleshy', 'submassive', 'round', 'massive',
       'tubular', 'bushy', 'external_polyps', 'foliose', 'solitary', 'brain',
       'phaceloid', 'branching', 'tabular', 'corymbose', 'lobed_brain',
       'cup_coral']
health_concept_columns = ['dead', 'bleached']

In [25]:
df_mapping_upd2 = df_mapping_upd.copy()
df_mapping_upd2[morphologic_concept_columns+health_concept_columns] = df_mapping_upd[morphologic_concept_columns+health_concept_columns].map(
    lambda x: -1 if (x is False or x=="False") else (0 if x == "not given" else 1)
)

In [26]:
df_mapping_upd2[morphologic_concept_columns+health_concept_columns]

,oval,arborescent,encrusting,digitate,meandroid,columnar,free_living,plating,fleshy,submassive,...,solitary,brain,phaceloid,branching,tabular,corymbose,lobed_brain,cup_coral,dead,bleached
0,-1,0,-1,0,-1,-1,-1,0,-1,-1,...,-1,-1,-1,1,0,0,-1,-1,-1,-1
1,-1,0,-1,0,-1,-1,-1,0,-1,-1,...,-1,-1,-1,1,0,0,-1,-1,-1,1
2,-1,0,-1,0,-1,-1,-1,0,-1,-1,...,-1,-1,-1,1,0,0,-1,-1,1,-1
3,0,0,-1,0,-1,0,0,0,0,-1,...,-1,0,-1,1,0,0,0,-1,-1,-1
4,-1,-1,-1,-1,1,0,0,0,0,1,...,-1,0,0,-1,-1,-1,1,-1,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1867,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1868,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1869,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1870,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [24]:
3**len(morphologic_concept_columns)

282429536481

In [27]:
def row_combo(row):
    return (
        tuple(sorted([col for col in morphologic_concept_columns+health_concept_columns if row[col] == 1])),
        tuple(sorted([col for col in morphologic_concept_columns+health_concept_columns if row[col] == 0])),
        tuple(sorted([col for col in morphologic_concept_columns+health_concept_columns if row[col] == -1])),
    )
morphologic_combinations = df_mapping_upd2[morphologic_concept_columns+health_concept_columns].apply(row_combo, axis=1)

unique_combinations = morphologic_combinations.drop_duplicates().reset_index(drop=True)
unique_combinations

0      ((branching,), (arborescent, bushy, corymbose,...
1      ((bleached, branching), (arborescent, bushy, c...
2      ((branching, dead), (arborescent, bushy, corym...
3      ((branching,), (arborescent, brain, bushy, col...
4      ((lobed_brain, massive, meandroid, submassive)...
                             ...                        
261    ((oval, round), (arborescent, bleached, brain,...
262    ((bleached, brain, lobed_brain, massive, meand...
263    ((massive, meandroid, submassive), (arborescen...
264    ((bleached, brain, massive, meandroid), (colum...
265    ((bleached, encrusting, plating), (arborescent...
Length: 266, dtype: object

In [20]:
unique_combinations

0      ((branching,), (arborescent, bushy, corymbose,...
1      ((branching,), (arborescent, brain, bushy, col...
2      ((lobed_brain, massive, meandroid, submassive)...
3      ((massive, submassive), (brain, bushy, columna...
4      ((brain, massive, submassive), (columnar, fles...
                             ...                        
207    ((branching, tabular), (arborescent, brain, bu...
208    ((columnar, plating), (arborescent, brain, bra...
209    ((branching, submassive), (arborescent, brain,...
210    ((oval, round), (arborescent, brain, branching...
211    ((massive, meandroid, submassive), (arborescen...
Length: 212, dtype: object